# 01 · Why Orchestration — Where the Agent Loop Breaks

> **Kernel:** `Phase 5 · core` · **Runs offline** — a scripted model makes every
> failure happen on purpose. One optional cell at the end repeats it with Ollama.

In Phase 3 you built an agent: a loop that asks the model what to do, runs the
tool it picks, feeds the result back, and repeats. That loop is a real agent —
and for a whole class of problems it is exactly the right tool.

This notebook is about the problems where it **isn't**. You will break the
Phase 3 loop in five specific, reproducible ways, and each break will point at
one idea that the rest of Phase 5 builds. By the end you'll know *why*
orchestration exists before you learn *how* any framework does it.

**You will learn**
- The two things a plain agent loop hides: **who decides control flow** and **where state lives**
- Five failure modes: *order*, *parallelism*, *durability*, *waiting for humans*, *inspect & replay*
- The four building blocks that fix them: **state, nodes, edges, checkpoints**
- When orchestration is overkill — and the price you pay for it

**Prerequisites:** Phase 3 notebooks 01–02 (agent loop), 12 (loop control), 23 (human-in-the-loop).

## 0 · Setup

Every Phase 5 notebook starts the same way: find the `PHASE_5_ORCHESTRATION`
folder, import `orch_lab` (which puts Phases 1–4 on the path), and print which
environment you're in.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'orch_lab').exists())
sys.path.insert(0, str(root))

import orch_lab
from orch_lab import ScriptedLLM, ProcessCrash, current_env
from agents_lab import Agent, ToolRegistry

print('environment:', current_env())

## 1 · Why this exists

Here is the heart of Phase 3's `Agent.run`, stripped to its skeleton:

```python
messages = [system, user_task]              # ← all state, in one local variable
for _ in range(max_steps):
    resp = llm.chat(messages, tools=...)    # ← the MODEL decides what happens next
    if not resp.tool_calls:
        return resp.text                    # model says "done"
    for call in resp.tool_calls:            # ← one call at a time
        result = tools.dispatch(call)
        messages.append(result)
# returning from run() → messages is garbage-collected
```

Read it with two questions in mind:

1. **Who decides what happens next?** The model, every single turn. Your code
   never says "after X, always do Y" — it only offers tools and hopes.
2. **Where does the work live?** In `messages`, a Python list in one process's
   RAM. When the function returns, or the process dies, the work is gone.

For a two-minute Q&A with a couple of tool calls, both are fine. Now imagine a
real job: *"Reproduce this bug, fix it, run the tests, wait for a reviewer to
approve, then merge."* That job needs guarantees:

| The job needs… | The plain loop gives… |
|---|---|
| Tests **always** run before merge | "The model will probably remember" |
| Three independent checks to run **at the same time** | One after another |
| Survive a crash at step 19 of 20 | Start again from step 1 (and pay again) |
| Wait **hours** for a reviewer | A blocked thread, or a denial |
| Rerun **just** the step that went wrong | Rerun everything |

Orchestration is the discipline of giving a multi-step AI system those
guarantees. Let's see each gap happen.

## 2 · How the demos work — a scripted model

A real model is non-deterministic: it *might* skip a step today and not
tomorrow. To show each failure reliably, we use `ScriptedLLM` — a fake
provider that replays a fixed list of turns. It speaks the exact same
interface as the `study_buddy` providers, so the **unchanged** Phase 3 `Agent`
runs on it.

A turn is either a string (the final answer) or a list of tool calls:

```python
ScriptedLLM([
    [{'name': 'run_tests', 'arguments': {}}],   # turn 1: call a tool
    'All good.',                                # turn 2: final answer
])
```

`ScriptedLLM` also counts calls and tokens (600 in / 60 out per call, a
realistic size for an agent turn), so we can measure wasted work.

> Scripting the model isn't cheating: every failure below is one real models
> produce in practice. The script just makes it happen on cue.

In [ ]:
def make_agent(turns, tools, **kwargs):
    '''The Phase 3 Agent with its model swapped for a scripted one.'''
    agent = Agent(provider='ollama', tools=tools, **kwargs)
    agent.llm = ScriptedLLM(turns)     # same interface as a real provider
    return agent

def show(result):
    for step in result.steps:
        print('  ', step)

## 3 · Five places the loop breaks

### 3.1 Order is a request, not a guarantee

**Rule:** never commit code unless the tests ran first. We put the rule in the
system prompt — the only place a plain loop *can* put it — and give the agent
both tools. The model (like real models on a long task) goes straight to
`commit`.

In [ ]:
calls = []
repo = ToolRegistry()

@repo.register(description='Run the test suite and report pass/fail.')
def run_tests():
    calls.append('run_tests')
    return '12 passed, 0 failed'

@repo.register(
    description='Commit the staged changes.',
    parameters={'type': 'object',
                'properties': {'message': {'type': 'string'}},
                'required': ['message']},
)
def commit(message):
    calls.append('commit')
    return f'committed: {message!r}'

SYSTEM = ('You are a release agent. RULE: you MUST call run_tests and see it '
          'pass BEFORE you call commit. Never commit untested code.')

agent = make_agent(
    [[{'name': 'commit', 'arguments': {'message': 'fix login bug'}}],
     'Committed the fix.'],
    tools=repo, system=SYSTEM,
)
result = agent.run('Ship the login bug fix.')
show(result)
print('\ntools actually executed, in order:', calls)

**What happened.** The rule lived in the prompt, and the prompt is *input to a
probabilistic model*, not a constraint on the program. Nothing in the loop
checks that `run_tests` happened — the loop runs whatever the model asks for.

**Could we patch it?** Sure: `approve_tool` could veto `commit` unless
`run_tests` is in the history. But then you're writing control flow ("B only
after A") inside a callback, scattered and invisible. Do that for ten rules and
you have an orchestrator — just a badly organised one.

**The idea it points to:** make order part of the **structure**. In a graph,
`commit` is reachable *only* through an edge from `run_tests`, so skipping it is
impossible rather than unlikely. → notebooks 04–05.

### 3.2 Everything runs one at a time

Three independent lookups, each taking 1 second (think: three API calls).
The model is well behaved: it asks for all three **in a single turn**, which
invites the program to run them together.

In [ ]:
import time

weather = ToolRegistry()

@weather.register(
    description='Current weather for a city (slow API, ~1s).',
    parameters={'type': 'object',
                'properties': {'city': {'type': 'string'}},
                'required': ['city']},
)
def lookup(city):
    time.sleep(1.0)
    return f'{city}: 21°C, clear'

three = [{'name': 'lookup', 'arguments': {'city': c}} for c in ('Pune', 'Oslo', 'Lima')]

agent = make_agent([three, 'Pune, Oslo and Lima are all clear.'], tools=weather)
t0 = time.perf_counter()
agent.run('Weather in Pune, Oslo and Lima?')
print(f'agent loop : {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
weather.run_parallel(three)          # the SAME registry, calls run concurrently
print(f'in parallel: {time.perf_counter() - t0:.1f}s')

**What happened.** The capability to run tools concurrently already existed —
Phase 3's `ToolRegistry.run_parallel` — yet the loop took ~3s, because the
loop's *shape* is `for call in resp.tool_calls: dispatch(call)`. How work runs
is decided by the control structure, not by the tools.

It gets worse when the model asks for the three lookups in three separate turns:
then you also pay for three model round-trips, each one re-reading the whole
conversation.

**The idea it points to:** **fan-out / fan-in** as a first-class shape — one
node splits work into branches that run at the same time, a later node joins
their results. → notebook 06.

### 3.3 A crash throws everything away

A research agent fetches three sources (each a paid model turn), then writes a
report. While writing, the process dies — a deploy, an out-of-memory kill, a
laptop lid closing. We simulate that with `ProcessCrash`, which (like a real
`kill -9`) can't be caught by the tool registry's `except Exception` safety net.

In [ ]:
fetches = []
crash_next_write = True
research = ToolRegistry()

@research.register(
    description='Fetch and summarise one source.',
    parameters={'type': 'object',
                'properties': {'topic': {'type': 'string'}},
                'required': ['topic']},
)
def fetch(topic):
    fetches.append(topic)
    return f'notes on {topic}'

@research.register(description='Write the final report from gathered notes.')
def write_report():
    global crash_next_write
    if crash_next_write:
        crash_next_write = False
        raise ProcessCrash('process killed while writing the report')
    return 'report.md written'

SCRIPT = [
    [{'name': 'fetch', 'arguments': {'topic': 'checkpointing'}}],
    [{'name': 'fetch', 'arguments': {'topic': 'retries'}}],
    [{'name': 'fetch', 'arguments': {'topic': 'fan-out'}}],
    [{'name': 'write_report', 'arguments': {}}],
    'Report is ready.',
]

attempt1 = make_agent(SCRIPT, tools=research)
try:
    attempt1.run('Research orchestration and write a report.')
except ProcessCrash as err:
    print('💥', err)
print(f'attempt 1: {attempt1.llm.calls} model calls, {attempt1.llm.tokens_used} tokens, fetched {fetches}')

Where are the three fetched notes now? They were in `messages`, a local variable
inside `run()`. The exception unwound that stack frame — they're gone. The only
way forward is to start over:

In [ ]:
attempt2 = make_agent(SCRIPT, tools=research)
result = attempt2.run('Research orchestration and write a report.')
print('final answer:', result.answer)

total_calls = attempt1.llm.calls + attempt2.llm.calls
wasted = attempt1.llm.calls
print(f'model calls: {total_calls} total, {wasted} of them repeated work '
      f'({wasted / total_calls:.0%})')
print('fetch() ran for:', fetches)

**What happened.** Every completed step was redone: the same fetches, the same
model turns, the same tokens. With a cloud model that's money; with slow tools
it's minutes or hours; and with tools that have **side effects** (send an email,
charge a card, open a PR) redoing them is actively harmful.

Note that *retries* don't fix this: a retry re-runs one failed call inside a
living process. **Durability** means the progress itself outlives the process.

**The idea it points to:** a **checkpoint** — after every step, save the state
*and* the position ("steps 1–3 done, next is 4") to durable storage. On restart,
load it and continue. → notebook 09, and Temporal in notebooks 44–45.

### 3.4 A human can't take their time

Phase 3 added `approve_tool`: before a risky tool runs, a callback asks a human.
But look at *where* that callback runs — inside `run()`, in the middle of the
loop. The loop cannot continue until the callback returns, so the program has
only two choices: **block** until the human answers, or **give up**.

Our reviewer is at lunch. We give them 1.5 seconds (in real life: a timeout of
minutes) and then deny.

In [ ]:
import threading

reviewer_answered = threading.Event()        # nobody will set this: they're at lunch

def ask_reviewer(tool, args):
    print(f'  ⏳ waiting for a reviewer to approve {tool}({args}) ...')
    return reviewer_answered.wait(timeout=1.5)   # True only if they answered in time

calls.clear()
agent = make_agent(
    [[{'name': 'run_tests', 'arguments': {}}],
     [{'name': 'commit', 'arguments': {'message': 'fix login bug'}}],
     'Could not commit: approval was not given.'],
    tools=repo, approve_tool=lambda t, a: t != 'commit' or ask_reviewer(t, a),
)
show(agent.run('Ship the login bug fix.'))

**What happened.** To wait longer, the process must stay alive, holding the
conversation in memory — for hours, across a lunch break, a deploy, a restart.
If it dies while waiting, you're back in 3.3. Real approvals (a code review, a
manager's sign-off, a customer's reply) take hours or days.

**The idea it points to:** make "waiting for a human" a **pause, not a block**.
Save a checkpoint, let the process exit, and resume from that checkpoint
whenever the answer arrives — even in a different process, on a different
machine. → notebook 10 (`interrupt`), and Temporal *signals* in notebook 45.

### 3.5 You can't inspect, test, or replay one step

Suppose the report in 3.3 came out wrong and you suspect the third fetch. You'd
like to (a) look at the exact state right before it, (b) change one thing, and
(c) re-run **from that point only**. Does the loop let you?

In [ ]:
import inspect
print('Agent.run', inspect.signature(Agent.run))
print()
print('What the run left behind — a log of text, not resumable state:')
show(result)

**What happened.** `run()` takes a task and nothing else: every run starts from
the beginning. The tracer gives you a readable log of *what happened*, but not
the state *at each point*, and there's no way to start from step 3. So:

- **Debugging** means re-running everything, with a non-deterministic model that
  may not even repeat the bug.
- **Testing** is all-or-nothing: there's no "step 3" to unit-test, because steps
  aren't named units — they're just iterations of a loop.
- **Auditing** ("why did it merge without approval?") means reading prose.

**The idea it points to:** named **nodes** you can test on their own, plus a
checkpoint at every step, which gives you **time travel**: load the state at
step 3, edit it, and fork a new run from there. → notebooks 04, 09 and 11.

## 4 · The fix in one picture: state + nodes + edges + checkpoints

Every failure above came from the same two things: the model decides the
control flow, and the work lives in RAM. Orchestration moves both into the open:

```
                 ┌────────────────────── State ──────────────────────┐
                 │ { task, notes: [...], tests_passed, approved, ...} │
                 └──────────────▲──────────────────────────┬─────────┘
                    each node   │ reads & returns updates  │ saved after every node
                                │                          ▼
   START ──► fetch_a ─┐                                 ┌──────────────┐
          ├► fetch_b ─┼──► write_report ──► run_tests ─►│ Checkpointer │
          └► fetch_c ─┘         ▲              │        │ (SQLite, …)  │
             (fan-out)          │   failed     │ passed └──────────────┘
                                └──────────────┤
                                               ▼
                                     human_approval (pause) ──► commit ──► END
```

| Building block | What it is | Fixes |
|---|---|---|
| **State** | One explicit, serialisable object every step reads and updates | 3.3, 3.5 — there's something to save and inspect |
| **Node** | A named step: plain code *or* an LLM/agent call | 3.5 — testable, replayable units |
| **Edge** | "After this node, go there" — fixed, conditional (a router), or a fan-out | 3.1, 3.2 — order and parallelism are structure |
| **Checkpointer** | Saves state + position after every node, keyed by a run id (*thread*) | 3.3, 3.4, 3.5 — resume, pause, time travel |

The model still makes the *judgement calls* — inside nodes, and in conditional
edges where you want it to choose. Orchestration only takes away the decisions
that should never have been left to chance.

Here's the smallest possible taste of it: an ordered, checkpointed runner in
fifteen lines. It's deliberately naive — notebooks 03–09 build the real engine.

In [ ]:
import json

CKPT = pathlib.Path('checkpoints/01_toy.json')
CKPT.parent.mkdir(exist_ok=True)
CKPT.unlink(missing_ok=True)                   # start fresh on every run of this cell

def run_workflow(steps, state, ckpt):
    '''Run steps in a FIXED order, saving state + progress after each one.'''
    saved = json.loads(ckpt.read_text()) if ckpt.exists() else {'done': [], 'state': state}
    for name, fn in steps:
        if name in saved['done']:
            print(f'  skip {name:<13} (checkpointed)')
            continue
        print(f'  run  {name}')
        saved['state'] = fn(saved['state'])
        saved['done'].append(name)
        ckpt.write_text(json.dumps(saved))    # ← durable progress
    return saved['state']

fetches.clear()
crash_next_write = True

def fetch_step(topic):
    def step(state):
        fetch(topic)                           # the same tool as in 3.3
        return {**state, 'notes': state['notes'] + [f'notes on {topic}']}
    return step

def report_step(state):
    return {**state, 'report': write_report()}  # crashes the first time

STEPS = [('fetch_ckpt',  fetch_step('checkpointing')),
         ('fetch_retry', fetch_step('retries')),
         ('fetch_fan',   fetch_step('fan-out')),
         ('write_report', report_step)]

print('attempt 1:')
try:
    run_workflow(STEPS, {'notes': []}, CKPT)
except ProcessCrash as err:
    print('  💥', err)

print('attempt 2 (a brand-new call, as if the process had restarted):')
final = run_workflow(STEPS, {'notes': []}, CKPT)
print('\nfinal state:', final)
print('fetch() ran for:', fetches, '← each source fetched once, not twice')

Two things to notice:

1. **Order is structural.** `write_report` comes after the fetches because the
   list says so — not because anyone was asked nicely.
2. **The state had to be JSON.** A checkpoint is written to disk and read back,
   possibly by a different process. Anything you keep in state must survive that
   round trip — which is why an open database connection or an SDK client never
   goes into orchestration state. You'll feel this constraint in every framework.

What the toy *can't* do yet: branch on a result, run the fetches in parallel,
let an LLM choose the next step, pause for a human, or keep separate runs apart.
That's the rest of Part A.

## 5 · Trade-offs — don't orchestrate everything

Orchestration isn't free. It trades the loop's flexibility for guarantees.

| Use a plain agent loop when… | Reach for orchestration when… |
|---|---|
| The task is short (seconds, a handful of steps) | The task runs for minutes, hours, or days |
| You can't predict the steps — open-ended exploration | Some steps and orderings are known in advance |
| Restarting from scratch is cheap and harmless | Steps are expensive, slow, or have side effects |
| One user, one process, interactive | Humans approve, or other systems call back later |
| You're prototyping | You must test, audit, and debug it in production |

**What you pay:**
- **More design up front.** You must decide the state schema and the steps
  before running anything.
- **Rigidity.** A fixed graph can't do what nobody drew. Good designs keep an
  agent *inside* a node for the open-ended parts.
- **Serialisation discipline.** Everything in state must be storable.
- **Another layer to learn and debug** — especially with a framework.

The rule of thumb that the whole phase keeps coming back to: **put determinism
where you need guarantees, and leave the model free where you need
judgement.** Notebook 02 turns that into a spectrum of designs.

## 6 · Common mistakes

- **Rules only in the prompt.** "Always run tests first" in a system prompt is a
  preference. If breaking it is unacceptable, make it an edge or a gate in code.
- **Asking the LLM to route what code can route.** If "failed tests → fix step"
  is a simple `if`, write the `if`. It's free, instant, and never wrong.
- **Confusing retries with durability.** Retrying a flaky call is good hygiene,
  but it doesn't survive the process dying. Only a checkpoint does.
- **Putting unserialisable objects in state** (clients, file handles, lambdas).
  It works until the first checkpoint, then fails. Keep *data* in state and
  *resources* outside it.
- **Adopting a framework before naming your problem.** Check which of the five
  gaps you actually have. If the answer is "none", a loop is the better design.

## 7 · How the frameworks name these ideas

Every framework in this phase is a different answer to the same five gaps.
You'll meet each in its own part; for now, just notice how each one describes
the **structure**:

| Framework | You describe the structure as… | Part |
|---|---|---|
| *from scratch* (`orch_lab`) | a graph of nodes and edges over a state dict | A |
| LangGraph | a `StateGraph` of nodes and edges over a typed state | B |
| OpenAI Agents SDK | agents that **hand off** to each other; the rest is your own Python | C |
| CrewAI | **crews** (agents + tasks) driven by event-based **Flows** | D |
| Microsoft Agent Framework | **workflows** of executors connected by edges | E |
| Google ADK | a tree of agents, with Sequential / Parallel / Loop workflow agents | F |
| Temporal | a deterministic **workflow** function whose side effects are **activities** | G |

Because you'll build the from-scratch version first, each framework will read
as "oh, that's their name for a node / an edge / a checkpoint".

## 8 · Try it live (optional, Ollama)

The scripted model forced the failure in 3.1. Does a real model respect the rule?
Run it a few times. It may well get it right — and that's the point: **"usually
right" is a probability, and a production rule needs a guarantee.**

In [ ]:
from study_buddy import get_provider

try:
    get_provider('ollama').client.list()        # is the Ollama server reachable?
    OLLAMA = True
except Exception as err:
    OLLAMA = False
    print('Ollama not reachable — skipping the live run.', err)

if OLLAMA:
    calls.clear()
    live = Agent(provider='ollama', tools=repo, system=SYSTEM, max_steps=4)
    show(live.run('The login fix is staged. Ship it — we are in a hurry, just commit.'))
    order_ok = 'commit' not in calls or ('run_tests' in calls and calls.index('run_tests') < calls.index('commit'))
    print('\ntools executed:', calls, '→', 'rule respected' if order_ok else 'RULE BROKEN')

## 9 · Check yourself

Try to answer before opening each one.

<details><summary><b>1.</b> Your agent must never call <code>deploy</code> unless <code>run_tests</code> passed in the same run. Where should that rule live, and why is the system prompt not enough?</summary>

In the **structure**: an edge that reaches `deploy` only from a "tests passed" branch (or, at minimum, a code gate before `deploy`). A prompt is input to a probabilistic model, so it makes a violation *unlikely*, not *impossible*. Anything with a hard "never" belongs in code.
</details>

<details><summary><b>2.</b> A 40-step research agent built on the Phase 3 loop crashes at step 38. What is lost, and what would you need to have saved to resume at step 38?</summary>

Everything: all tool results and model turns lived in the local `messages` list. To resume you need, after every step, the **state** (messages / gathered results) *and* the **position** (what runs next), written to durable storage and keyed by a run id. That is exactly what a checkpointer stores.
</details>

<details><summary><b>3.</b> Why did three 1-second lookups take ~3s in the loop, even though the tool registry has <code>run_parallel</code>?</summary>

Because concurrency is a property of the **control flow**, not of the tools. The loop is written as `for call in tool_calls: dispatch(call)`, so it runs calls one by one no matter what the registry can do. Parallelism has to be designed into the structure (fan-out / fan-in).
</details>

<details><summary><b>4.</b> Why is "retry the failed call three times" not the same as durability?</summary>

A retry happens *inside a running process* and only covers one call. If the process itself dies — deploy, OOM, crash — the retry logic dies with it, and so does all progress. Durability means progress is saved outside the process, so a *new* process can continue.
</details>

<details><summary><b>5.</b> Name two situations where adding orchestration would be the wrong call.</summary>

For example: a chat assistant answering a question with one or two tool calls; an exploratory prototype whose steps you can't predict yet; any short task where restarting is cheap and has no side effects. There the loop's flexibility is worth more than the guarantees.
</details>

<details><summary><b>6.</b> In the toy runner, why would storing an open SQLite connection inside <code>state</code> break the design?</summary>

State is written to disk (here as JSON) and read back, possibly by a different process. A connection is a live resource, not data: it can't be serialised, and even if it could, it would be meaningless in another process. Keep data in state and resources outside it.
</details>

## Takeaway

A plain agent loop hides two decisions: **the model owns the control flow**, and
**the work lives in one process's memory**. That's why it can't guarantee order,
run things in parallel, survive a crash, wait for a human, or replay a step.
Orchestration makes both explicit — **state, nodes, edges, checkpoints** — while
leaving the model in charge of the judgement calls inside them.

Next → `02_workflows_vs_agents.ipynb`: the spectrum from fixed workflows to fully
autonomous agents, and how to pick the right point on it.